# Accompanying Code for "Strategies to Avoid Poor Local Minima in Nonlinear Optimization"
**Submission date**: 09.06.2026

**Author**: Nicolas Heidbrink

**Supervisor**: Stefan Schwarze, M. Sc.

**GitHub repository**: https://github.com/nicolasheidbrink/optimization-seminar


*Google Gemini was used for debugging and optimizing code structure, formatting, and documentation.*

## 0. Imports

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
from scipy.optimize import minimize
from contextlib import redirect_stdout
from tunneling_algorithm import TunnelingAlgorithm
import functions as fn
from algo_tester import Algorithm_Tester


## 1. The tunneling algorithm missing global minima
As mentioned in the seminar paper, the tunneling algorithm can stagnate when the movable pole gets closer and closer to the current iterate. This can be seen by examining the logs of an application of the tunneling function on the three-dimensional shubert function:


In [2]:
np.random.seed(42)
with open("shubert_3d_logs.txt", "w") as f:
    with redirect_stdout(f):
        ta = TunnelingAlgorithm(fn.shubert, fn.altered_shubert, [[-10, 10]]*3, verbose=True)
        x_stars, f_star = ta.apply_algorithm(np.array([np.random.uniform(-10, 10)]*3))
print(f"The tunneling algorithm found {len(x_stars)} points with the function value {f_star}")

The tunneling algorithm found 1 points with the function value -1459.0944466182596


Between lines 943 and 1168, the iterates seem to stagnate around $(-7.22, -6.43, -7.48)^T$. In the 25 iterations starting at line 943, the displacements go from being `[-0.00495506  0.02873972 -0.00575734]` to `[-1.56197278e-10  8.57574456e-10 -1.80335303e-10]`, a change equivalent to element-wise multiplication of the first displacement vector with `[3.15227824e-08 2.98393462e-08 3.13226773e-08]`.

 According to the reasoning in the seminar paper and assuming the algorithm ran into the described trap, the expected factor to multiply the first iterate vector with to get the second iterate vector would be $\left( \frac{1}{2\lambda_0} \right) ^ {25} = 2.980232 \times 10^{-8}$, which closely matches the observed results. (The logs confirm that $\lambda_0 = 1$ for the entire sequence examined.)

 Looking at how the displacements change from one iterate to the next between lines 943 and 1168 also reveals that they are approximately halved with each iteration.

## 2. Improving the tunneling algorithm
To combat the problem of movable poles getting so close to tunneling phase iterates that the displacements gets extremely small, as mentioned in the seminar paper, one logical solution was to fix the movable pole's distance from iterates. Levy and Montalvo state that the movable pole should always be in the interior of a unit ball around the current iterate, so the distance was fixed to $0.9$. 

Also, a change was made to minimize the risk of random perturbations at the start of tunneling phases restricting the areas searched: These random perturbations now point generally in both directions down each axis of the domain. The changes were implemented in $\texttt{tunneling\_algorithm.py}$ and are activated by setting $\texttt{TunnelingAlgorithm.improved}$ to $\texttt{True}$ while initializing the object.

The effect of the changes can be examined by testing both versions of the algorithm with $\texttt{algo\_tester.py}$, a script that runs the algorithms on a variety of functions and records whether and how many of their global minimal points were found in how much time.

In [3]:
algorithm_tester = Algorithm_Tester()
algorithm_tester.run_tests(TunnelingAlgorithm, improved=False)

Test: Shubert 2D, average number of minima found: 1.52 out of 18, failure rate: 17.50% over 40 trials, time per trial: 0.467 CPU seconds.
Test: Shubert 3D, average number of minima found: 0.40 out of 81, failure rate: 60.00% over 10 trials, time per trial: 0.458 CPU seconds.
Test: Shubert 6D, average number of minima found: 0.00 out of 4374, failure rate: 100.00% over 10 trials, time per trial: 0.711 CPU seconds.
Test: Six Hump Camel, average number of minima found: 1.47 out of 2, failure rate: 0.00% over 30 trials, time per trial: 0.082 CPU seconds.
Test: Altered Shubert 2D, average number of minima found: 0.13 out of 1, failure rate: 86.67% over 30 trials, time per trial: 0.341 CPU seconds.
Test: Function 15 2D, average number of minima found: 1.00 out of 1, failure rate: 0.00% over 30 trials, time per trial: 0.132 CPU seconds.
Test: Function 15 3D, average number of minima found: 0.85 out of 1, failure rate: 15.00% over 20 trials, time per trial: 0.233 CPU seconds.
Test: Function 15

In [4]:
algorithm_tester.run_tests(TunnelingAlgorithm, improved=True)

Test: Shubert 2D, average number of minima found: 13.68 out of 18, failure rate: 0.00% over 40 trials, time per trial: 1.834 CPU seconds.
Test: Shubert 3D, average number of minima found: 13.30 out of 81, failure rate: 0.00% over 10 trials, time per trial: 1.031 CPU seconds.
Test: Shubert 6D, average number of minima found: 3.80 out of 4374, failure rate: 60.00% over 10 trials, time per trial: 1.303 CPU seconds.
Test: Six Hump Camel, average number of minima found: 2.00 out of 2, failure rate: 0.00% over 30 trials, time per trial: 0.206 CPU seconds.
Test: Altered Shubert 2D, average number of minima found: 0.93 out of 1, failure rate: 6.67% over 30 trials, time per trial: 0.506 CPU seconds.
Test: Function 15 2D, average number of minima found: 1.00 out of 1, failure rate: 0.00% over 30 trials, time per trial: 0.158 CPU seconds.
Test: Function 15 3D, average number of minima found: 1.00 out of 1, failure rate: 0.00% over 20 trials, time per trial: 0.243 CPU seconds.
Test: Function 15 4D

Clearly, the results with the improvements are superior, although they still do not match Levy and Montalvo's results.

## 3. Implementing simulated annealing
As described in the seminar paper, simulated annealing was implemented under the flag `annealing` in `tunneling_algorithm.py`. The tests are run here.

In [5]:
algorithm_tester.run_tests(TunnelingAlgorithm, annealing=True)

Test: Shubert 2D, average number of minima found: 10.85 out of 18, failure rate: 0.00% over 40 trials, time per trial: 0.877 CPU seconds.
Test: Shubert 3D, average number of minima found: 5.10 out of 81, failure rate: 0.00% over 10 trials, time per trial: 1.105 CPU seconds.
Test: Shubert 6D, average number of minima found: 0.10 out of 4374, failure rate: 90.00% over 10 trials, time per trial: 1.267 CPU seconds.
Test: Six Hump Camel, average number of minima found: 1.50 out of 2, failure rate: 0.00% over 30 trials, time per trial: 0.118 CPU seconds.
Test: Altered Shubert 2D, average number of minima found: 0.67 out of 1, failure rate: 33.33% over 30 trials, time per trial: 0.388 CPU seconds.
Test: Function 15 2D, average number of minima found: 1.00 out of 1, failure rate: 0.00% over 30 trials, time per trial: 0.421 CPU seconds.
Test: Function 15 3D, average number of minima found: 0.95 out of 1, failure rate: 5.00% over 20 trials, time per trial: 1.698 CPU seconds.
Test: Function 15 4D